# Christina Doty

In [ ]:
import pandas as pd
import numpy as np
import os
import pathlib as Path
from sklearn.model_selection import train_test_split

# Tidy Crop Data

In [ ]:
def tidy_crop(read_path, write_path, crop='wheat', granularity='state'):
    if granularity == 'state':
        keep_cols = ["year", "state", "state_abbr", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "commodity"]
    elif granularity == 'county':
        keep_cols = ["year", "state", "state_abbr", "county", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "county", "commodity"]
    else:
        print("granularity should be either state or county")
        return None
    
    if crop == 'corn':
        keep_cols.append("util_practice_desc")

    df = pd.read_csv(read_path)
    # remove the duplicated yield rows that have dollar amounts instead of BU
    df_no_money = df.loc[df['unit'] != '$']
    df_usefulcols = df_no_money[keep_cols]

    if crop == 'wheat':
        df_pivot = df_usefulcols.pivot_table(index=pivot_index_cols, columns="statistic_category", values="value", aggfunc="first")
        df_pivot.columns.name=None
        df_tidy = df_pivot.reset_index()
        print(f"Sanity check, this should be 4: {len(df_usefulcols) / len(df_tidy)}")
    elif crop == 'corn':
        # Split into planted and harvest groups
        df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
        df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

        # For harvest rows, combine the stat category and util practice into one label
        # # e.g. "area harvested grain", "production silage", "yield grain" etc.
        df_harvest = df_harvest.copy()
        df_harvest["stat_label"] = (df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower())
        
        # remove the extra area planted entries that appear for the grain category
        df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

        # For planted rows, the label is just the stat category
        df_planted = df_planted.copy()
        df_planted["stat_label"] = df_planted["statistic_category"]

        # Combine and pivot once on the new label
        df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

        df_tidy = df_combined.pivot_table(index=pivot_index_cols, columns="stat_label", values="value", aggfunc="first").reset_index()
        df_tidy.columns.name = None
    else:
        print("crop should be either wheat or corn")
        return None
    
    df_tidy.to_csv(write_path)
    return df_tidy

In [ ]:
wheat_state_tidy = tidy_crop("../data/wheat_state.csv", "../data/wheat_state_tidy.csv", crop='wheat', granularity='state')
print(len(wheat_state_tidy))
wheat_state_tidy.head(5)

In [ ]:
wheat_county_tidy = tidy_crop("../data/wheat_county.csv", "../data/wheat_county_tidy.csv", crop='wheat', granularity='county')
print(len(wheat_county_tidy))
wheat_county_tidy.head(5)

In [ ]:
corn_state_tidy = tidy_crop("../data/corn_state.csv", "../data/corn_state_tidy.csv", crop='corn', granularity='state')
print(len(corn_state_tidy))
corn_state_tidy.head(5)

In [ ]:
corn_county_tidy = tidy_crop("../data/corn_county.csv", "../data/corn_county_tidy.csv", crop='corn', granularity='county')
print(len(corn_county_tidy))
corn_county_tidy.head(5)

# Tidy emissions data

In [ ]:
def tidy_emissions(read_path, write_path):
    df = pd.read_csv(read_path)
    df_agr = df[df["sector"] == "agriculture"].copy()
    
    subsector_names = {
        "crop-residues": "crop_residues",
        "enteric-fermentation-cattle-pasture": "livestock",
        "enteric-fermentation-cattle-operation": "livestock",
        "enteric-fermentation-other": "livestock",
        "manure-left-on-pasture-cattle": "livestock",
        "manure-applied-to-soils": "manure_fertilizer",
        "manure-management-cattle-operation": "livestock",
        "manure-management-other": "livestock",
        "synthetic-fertilizer-application": "synth_fertilizer",
        "other-agricultural-soil-emissions": "soil",
        "rice-cultivation": "rice",
        "cropland-fires": "crop_fire"
    }

    df_agr["source"] = [subsector_names[subsector] for subsector in df_agr["subsector"]]
    df_agr = df_agr[df_agr["source"] != "livestock"][["year", "state", "admin", "source", "gas", "emissionsQuantity"]]

    df_agr = df_agr.copy()
    df_agr["emission"] = (df_agr["source"] + "_" + df_agr["gas"].str.lower())

    df_pivot = df_agr.pivot_table(index=["year", "state", "admin"], columns="emission", values="emissionsQuantity", aggfunc="sum").reset_index()
    df_pivot.columns.name = None

    df_pivot.to_csv(write_path)
    return df_pivot

read_path = "../data/ct_match_corn_ALL_subsectors_2021_2024.csv"

emissions_county_tidy = tidy_emissions("../data/ct_match_corn_ALL_subsectors_2021_2024.csv", "../data/climateTRACE_tidy.csv")
emissions_county_tidy.head(5)

# Clean and Combine Dataframes

In [ ]:
def clean_data(wheat_file, corn_file, emissions_file, write_path):
    wheat_df = pd.read_csv(wheat_file).drop(columns=['Unnamed: 0', 'state_abbr', 'commodity'], axis=1)
    corn_df = pd.read_csv(corn_file).drop(columns=['Unnamed: 0', 'state_abbr', 'commodity'], axis=1)
    emissions_df = pd.read_csv(emissions_file).drop(columns=['Unnamed: 0'], axis=1)

    wheat_df = wheat_df.rename(columns={
        "year": "Year",
        "state": "State",
        "county": "County",
        "AREA HARVESTED": "Wheat Harvested",
        "AREA PLANTED": "Wheat Planted",
        "PRODUCTION": "Wheat Production",
        "YIELD": "Wheat Yield"
        })

    corn_df = corn_df.rename(columns={
        "year": "Year",
        "state": "State",
        "county": "County",
        "AREA HARVESTED grain": "Corn Grain Harvested",
        "AREA HARVESTED silage": "Corn Silage Harvested",
        "AREA PLANTED": "Corn Planted",
        "PRODUCTION grain":"Corn Grain Production",
        "PRODUCTION silage":"Corn Silage Production",
        "YIELD grain": "Corn Grain Yield",
        "YIELD silage": "Corn Silage Yield"
        })

    emissions_df = emissions_df.rename(columns={
        "year": "Year",
        "state": "State",
        "admin": "County",
        "crop_fire_co2e_100yr": "Crop Fire CO2",
        "crop_fire_n2o": "Crop Fire N2O",
        "crop_residues_co2e_100yr": "Crop Residue CO2",
        "crop_residues_n2o": "Crop Residue N2O",
        "manure_fertilizer_co2e_100yr": "Manure Fertilizer CO2",
        "manure_fertilizer_n2o": "Manure Fertilizer N2O",
        "rice_co2e_100yr": "Rice CO2",
        "rice_n2o": "Rice N2O",
        "soil_co2e_100yr": "Soil CO2",
        "soil_n2o": "Soil N2O",
        "synth_fertilizer_co2e_100yr": "Synthetic Fertilizer CO2",
        "synth_fertilizer_n2o": "Synthetic Fertilizer N2O"
        })
    
    corn_df = corn_df[corn_df["Year"].isin([2021,2022, 2023])].copy()
    emissions_df = emissions_df[emissions_df["Year"].isin([2021,2022, 2023])].copy()

    df_merged = (
        emissions_df
        .merge(
            corn_df,
            on=["Year", "State", "County"],
            how="left"
        )
        .merge(
            wheat_df,
            on=["Year", "State", "County"],
            how="left"
        )
    )

    df_merged = df_merged.dropna(subset=["Corn Planted", "Corn Grain Harvested", "Corn Silage Harvested", "Wheat Planted"], how="all").copy()

    crop_cols = df_merged.filter(regex="Wheat|Corn").columns
    df_merged[crop_cols] = df_merged[crop_cols].fillna(0)

    df_merged.to_csv(write_path, index=False)
    return df_merged

merged_df = clean_data("../data/wheat_county_tidy.csv", "../data/corn_county_tidy.csv", "../data/climateTRACE_tidy.csv", "../data/clean_data.csv")


In [ ]:
def create_splits(read_path, write_folder):
    df = pd.read_csv(read_path)
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42, stratify=df["Year"])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["Year"])

    train_df.to_csv(os.path.join(write_folder, "train.csv"), index=False)
    val_df.to_csv(os.path.join(write_folder, "validation.csv"), index=False)
    test_df.to_csv(os.path.join(write_folder, "test.csv"), index=False)
    return train_df, val_df, test_df

train, val, test = create_splits("../data/clean_data.csv", "../data")

# Scratch Work Below

In [ ]:
raw_df = pd.read_csv("../data/corn_state.csv")
raw_df.head(2)

In [ ]:
df_no_money = raw_df.loc[raw_df['unit'] != '$']

In [ ]:
df_usefulcols = df_no_money[["year", "state", "state_abbr", "commodity", "util_practice_desc", "statistic_category", "value"]]
df_usefulcols.head(2)

In [ ]:
df_pivot = df_usefulcols.pivot_table(index=["year", "state", "state_abbr", "util_practice_desc", "commodity"], columns="statistic_category", values="value", aggfunc="first")
df_pivot.columns.name=None
df_pivot = df_pivot.reset_index()
df_pivot.head(5)

In [ ]:
############ Doesn't work

# Split into the three groups
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# Pivot each separately
planted_pivot = df_planted.pivot_table(index=["year", "state", "state_abbr", "commodity"], columns="statistic_category", values="value", aggfunc="first").reset_index()
planted_pivot.columns.name = None

# Will only have 'area planted', rename to be explicit
#planted_pivot = planted_pivot.rename(columns={"area planted": "area planted"})

harvest_pivot = df_harvest.pivot_table(
    index=["year", "state", "state_abbr", "commodity", "util_practice_desc"],
    columns="statistic_category",
    values="value",
    aggfunc="first"
).reset_index()
harvest_pivot.columns.name = None

# Merge area planted onto each grain/silage row
df_final = harvest_pivot.merge(
    planted_pivot[["year", "state", "state_abbr", "commodity", "AREA PLANTED"]],
    on=["year", "state", "state_abbr", "commodity"],
    how="left"
)
df_final.head(10)

In [ ]:
# Split into planted and harvest groups as before
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# For harvest rows, combine the stat category and util practice into one label
df_harvest = df_harvest.copy()
df_harvest["stat_label"] = (
    df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower()
)
# e.g. "area harvested grain", "production silage", "yield grain" etc.

df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

# For planted rows, the label is just the stat category + total
df_planted = df_planted.copy()
df_planted["stat_label"] = df_planted["statistic_category"] + " total"
# e.g. "area planted total"

# Combine and pivot once on the new label
df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

df_final = df_combined.pivot_table(
    index=["year", "state"],
    columns="stat_label",
    values="value",
    aggfunc="first"
).reset_index()
df_final.columns.name = None

In [ ]:
df_final.head(10)

In [ ]:
len(df_usefulcols) / len(df_pivot)

In [ ]:
index_cols = ["year", "state", "statistic_category"]

duplicates = raw_df[raw_df.duplicated(subset=index_cols, keep=False)]
#print(duplicates.sort_values(index_cols))

In [ ]:
print(duplicates[["description", "class", "statistic_category"]].drop_duplicates())

### Tidy climate trace data

In [ ]:
read_path = "../data/Copy of ct_match_corn_IOWA_subsectors_2021_2024.csv"
df = pd.read_csv(read_path)
df.head(10)

In [ ]:
df_agr = df[df["sector"] == "agriculture"].copy()
df_agr.head(10)

In [ ]:
len(df)/len(df_agr)

In [ ]:
print(len(set(df_agr["subsector"].tolist())))
print(list(set(df_agr["subsector"].tolist())))

In [ ]:
subsector_names = {
    "crop-residues": "crop_residues",
    "enteric-fermentation-cattle-pasture": "livestock",
    "enteric-fermentation-cattle-operation": "livestock",
    "enteric-fermentation-other": "livestock",
    "manure-left-on-pasture-cattle": "livestock",
    "manure-applied-to-soils": "manure_fertilizer",
    "manure-management-cattle-operation": "livestock",
    "manure-management-other": "livestock",
    "synthetic-fertilizer-application": "synth_fertilizer",
    "other-agricultural-soil-emissions": "soil",
    "rice-cultivation": "rice",
    "cropland-fires": "crop_fire"
}

df_agr["source"] = [subsector_names[subsector] for subsector in df_agr["subsector"]]
df_agr = df_agr[df_agr["source"] != "livestock"][["year", "state", "admin", "source", "gas", "emissionsQuantity"]]
df_agr.head(10)

In [ ]:
df_agr = df_agr.copy()
df_agr["emission"] = (
    df_agr["source"] + "_" + df_agr["gas"].str.lower()
)
df_agr.head(10)

In [ ]:
df_pivot = df_agr.pivot_table(
    index=["year", "state", "admin"],
    columns="emission",
    values="emissionsQuantity",
    aggfunc="sum"
).reset_index()
df_pivot.columns.name = None

In [ ]:
df_pivot.head(10)

### Combine data frames

In [ ]:
wheat_df = pd.read_csv("../data/wheat_county_tidy.csv").drop(columns=['Unnamed: 0', 'state_abbr', 'commodity'], axis=1)
corn_df = pd.read_csv("../data/corn_county_tidy.csv").drop(columns=['Unnamed: 0', 'state_abbr', 'commodity'], axis=1)
emissions_df = pd.read_csv("../data/climateTRACE_tidy.csv").drop(columns=['Unnamed: 0'], axis=1)

wheat_df = wheat_df.rename(columns={
    "year": "Year",
    "state": "State",
    "county": "County",
    "AREA HARVESTED": "Wheat Harvested",
    "AREA PLANTED": "Wheat Planted",
    "PRODUCTION": "Wheat Production",
    "YIELD": "Wheat Yield"
    })

corn_df = corn_df.rename(columns={
    "year": "Year",
    "state": "State",
    "county": "County",
    "AREA HARVESTED grain": "Corn Grain Harvested",
    "AREA HARVESTED silage": "Corn Silage Harvested",
    "AREA PLANTED": "Corn Planted",
    "PRODUCTION grain":"Corn Grain Production",
    "PRODUCTION silage":"Corn Silage Production",
    "YIELD grain": "Corn Grain Yield",
    "YIELD silage": "Corn Silage Yield"
    })

emissions_df = emissions_df.rename(columns={
    "year": "Year",
    "state": "State",
    "admin": "County",
    "crop_fire_co2e_100yr": "Crop Fire CO2",
    "crop_fire_n2o": "Crop Fire N2O",
    "crop_residues_co2e_100yr": "Crop Residue CO2",
    "crop_residues_n2o": "Crop Residue N2O",
    "manure_fertilizer_co2e_100yr": "Manure Fertilizer CO2",
    "manure_fertilizer_n2o": "Manure Fertilizer N2O",
    "rice_co2e_100yr": "Rice CO2",
    "rice_n2o": "Rice N2O",
    "soil_co2e_100yr": "Soil CO2",
    "soil_n2o": "Soil N2O",
    "synth_fertilizer_co2e_100yr": "Synthetic Fertilizer CO2",
    "synth_fertilizer_n2o": "Synthetic Fertilizer N2O"
    })

In [ ]:
wheat_df.head(1)

In [ ]:
corn_df.head(1)

In [ ]:
emissions_df.head(1)

In [ ]:
# add 'wheat', 'corn', and 'emit' to all columns
# merge on year+state+county

In [ ]:
print(sorted(list(set(wheat_df["Year"]))))
print(sorted(list(set(corn_df["Year"]))))
print(sorted(list(set(emissions_df["Year"]))))

In [ ]:
# Note: USDA faced budget cuts so we don't have county level wheat data after 2023
# Also note: we have month by month usda data as well, which we could use, but that might complicate the model too much

In [ ]:
corn_df = corn_df[corn_df["Year"].isin([2021,2022, 2023])].copy()
emissions_df = emissions_df[emissions_df["Year"].isin([2021,2022, 2023])].copy()

In [ ]:
print(sorted(list(set(wheat_df["Year"]))))
print(sorted(list(set(corn_df["Year"]))))
print(sorted(list(set(emissions_df["Year"]))))

In [ ]:
df_merged = (
    emissions_df
    .merge(
        corn_df,
        on=["Year", "State", "County"],
        how="left"
    )
    .merge(
        wheat_df,
        on=["Year", "State", "County"],
        how="left"
    )
)

In [ ]:
corn_na = corn_df[corn_df['Corn Grain Harvested'].isna() & corn_df['Corn Silage Harvested'].isna()]
corn_na

In [ ]:
print(f"Emissions rows:  {len(emissions_df)}")
print(f"Merged rows:     {len(df_merged)}")
print(f"Missing corn:    {(df_merged['Corn Planted'].isna() & df_merged['Corn Grain Harvested'].isna() & df_merged['Corn Silage Harvested'].isna()).sum()}")
print(f"Missing wheat:   {df_merged['Wheat Planted'].isna().sum()}")
print(f"Missing both:    {(df_merged['Corn Planted'].isna() & df_merged['Wheat Planted'].isna()).sum()}")

In [ ]:
df_merged = df_merged.dropna(subset=["Corn Planted", "Corn Grain Harvested", "Corn Silage Harvested", "Wheat Planted"], how="all")

In [ ]:
print(f"Merged rows:     {len(df_merged)}")

In [ ]:
df_merged.head(2)

In [ ]:
crop_cols = df_merged.filter(regex="Wheat|Corn").columns
df_merged[crop_cols] = df_merged[crop_cols].fillna(0)
df_merged.head(2)

In [ ]:
print(df_merged.isna().sum())

In [ ]:
print(len(df_merged[df_merged["Year"] == 2021]))
print(len(df_merged[df_merged["Year"] == 2022]))
print(len(df_merged[df_merged["Year"] == 2023]))